In [1]:
#!pip uninstall huetracer -y
#!pip install huetracer
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
import numpy as np
import pandas as pd
import huetracer
import scvi
import gc
import math
import bin2cell as b2c
import torch
from itertools import cycle
import warnings
import importlib
import sys

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
pd.set_option('display.max_columns', None)
sc.set_figure_params(figsize=[10,10],dpi=100)

scvi.settings.seed = 0
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else"mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
device_str = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
warnings.filterwarnings("ignore", message=".*must be within the support of the distribution.*")

Seed set to 0


Using device: mps


In [2]:
### parameters to be input
SAMPLE_NAME = 'E16_15'
lib_id = SAMPLE_NAME # list(sp_adata.uns['spatial'].keys())[0]

path = os.path.expanduser("~")+"/Desktop/space/E16_15"
save_path_for_today = os.path.expanduser("~")+"/tmp/outputs/250610_" + SAMPLE_NAME
visium_path = "/Volumes/Public/data/tendon_mouse/visium_HD/"
source_image_path = visium_path + "he/" + SAMPLE_NAME + ".tif"
expression_path = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_002um"
### optional 8/16 um binned dataset
expression_path_8um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_008um"
expression_path_16um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_016um"
###

# area to be analyzed
# ## GCTB spatial G1, FFPE9
mask_large_x1, mask_large_x2, mask_large_y1, mask_large_y2 = 450, 1950, 250, 1750

# Species = "Human"
Species = "Mouse"

# List of target gene names
if Species == "Human":
    target_genes = [
        'TNFSF11' # Add more genes here if needed
    ]
    prefix_mt = 'MT-'
else:
    target_genes = [
        'Col1a1' # Add more genes here if needed
    ]
    prefix_mt = 'mt-'


# setting for filenames
label_image_filename = "he_labels_image.pdf"
h5ad_filename = SAMPLE_NAME + "_b2c.h5ad"
h5ad_full_filename = SAMPLE_NAME + "_2um.h5ad"
h5ad_predicted_full_filename = SAMPLE_NAME + "_nucleus_predicted.h5ad"
h5ad_sc_filtered_full_filename = SAMPLE_NAME + "_single_cell_filtered.h5ad"
h5ad_sc_microenvironment_full_filename = SAMPLE_NAME + "_single_cell_microenvironment.h5ad"
save_spatial_plot_path = os.path.join(save_path_for_today, "cropped_spatial_plot.svg")
save_svg_path = os.path.join(save_path_for_today, "spatial_salvage_labels.svg")
h5ad_save_path = os.path.join(save_path_for_today, h5ad_filename)
h5ad_full_save_path = os.path.join(save_path_for_today, h5ad_full_filename)
h5ad_predicted_full_save_path = os.path.join(save_path_for_today, h5ad_predicted_full_filename)
h5ad_sc_filtered_full_save_path = os.path.join(save_path_for_today, h5ad_sc_filtered_full_filename)
h5ad_microenvironment_full_save_path = os.path.join(save_path_for_today, h5ad_sc_microenvironment_full_filename)

os.chdir(path)
os.makedirs(save_path_for_today, exist_ok=True)


In [ ]:
# Spatial data load and curation
sp_adata_raw = sc.read_h5ad(h5ad_save_path)
cell_mask = ((sp_adata_raw.obs['array_row'] >= mask_large_x1) & 
             (sp_adata_raw.obs['array_row'] <= mask_large_x2) & 
             (sp_adata_raw.obs['array_col'] >= mask_large_y1) & 
             (sp_adata_raw.obs['array_col'] <= mask_large_y2)
            )
sp_adata = sp_adata_raw[cell_mask].copy()
sp_adata.var_names_make_unique()
sp_adata = sp_adata[sp_adata.obs['bin_count']>5].copy() # min 6 bins
#need integers for seurat v3 hvgs
sp_adata.X = np.round(sp_adata.X).copy()
sp_adata.raw = sp_adata.copy()
sp_adata.var['MT'] = sp_adata.var_names.str.startswith(prefix_mt)
sc.pp.calculate_qc_metrics(sp_adata, qc_vars=['MT'], percent_top=None, log1p=False, inplace=True)
sc.pl.highest_expr_genes(sp_adata, n_top=30)
sc.pl.violin(sp_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_MT'], jitter=0.4, multi_panel=True)
sc.pl.scatter(sp_adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_MT', size=40)
sc.pl.scatter(sp_adata, x='total_counts', y='pct_counts_MT')
fig = plt.figure()
sns.displot(sp_adata.obs['pct_counts_MT'][sp_adata.obs['pct_counts_MT'] < 10], kde=False)
plt.show()

print('Total number of cells: {:d}'.format(sp_adata.n_obs))

# Filtering
# sc.pp.filter_cells(sp_adata, min_genes=40)
# sc.pp.filter_genes(sp_adata, min_cells=100)
# print('Number of cells after low-quality cell filter: {:d}'.format(sp_adata.n_obs))

sc.pp.filter_cells(sp_adata, min_counts = 200)
#sc.pp.filter_cells(sp_adata, max_counts = 2000)
print('Number of cells after count filter: {:d}'.format(sp_adata.n_obs))

sp_adata = sp_adata[sp_adata.obs['pct_counts_MT'] < 10]
print('Number of cells after MT filter: {:d}'.format(sp_adata.n_obs))

sc.pl.highest_expr_genes(sp_adata, n_top=30)
sc.pl.violin(sp_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_MT'], jitter=0.4, multi_panel=True)
sc.pl.scatter(sp_adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_MT', size=40)
sc.pl.scatter(sp_adata, x='total_counts', y='pct_counts_MT')
fig = plt.figure()
sns.displot(sp_adata.obs['pct_counts_MT'][sp_adata.obs['pct_counts_MT'] < 10], kde=False)
plt.show()
# save raw count data
sp_adata.layers["counts"] = sp_adata.X.copy()
sp_adata.raw = sp_adata

# reset with raw count data
sp_adata_sponly = sp_adata.copy()
sp_adata_sponly.X = sp_adata_sponly.raw.X.copy()
sp_adata_sponly.var = sp_adata_sponly.raw.var.copy()
sp_adata_sponly.layers["counts"] = sp_adata_sponly.X.copy()
sc.pp.normalize_total(sp_adata_sponly)
sc.pp.log1p(sp_adata_sponly)
sc.pp.highly_variable_genes(sp_adata_sponly,
                            flavor = 'seurat_v3',
                            n_top_genes=2000,
                            layer = "counts",
                            subset = False)
sc.tl.pca(sp_adata_sponly, svd_solver='arpack',mask_var='highly_variable', n_comps=20)
sc.pp.neighbors(sp_adata_sponly, random_state=SEED)
sc.tl.umap(sp_adata_sponly, random_state=SEED)
sc.tl.leiden(sp_adata_sponly, resolution=1, flavor="igraph", n_iterations=2, key_added='leiden', random_state=SEED)
sc.pl.pca(sp_adata_sponly, color='total_counts', components=['1,2', '2,3', '1,3'])
sc.pl.pca_variance_ratio(sp_adata_sponly, log=True)
sc.pl.umap(sp_adata_sponly, color=['leiden', 'total_counts'],use_raw=False)
sc.pl.spatial(sp_adata_sponly, color='leiden',
              title='Annotation for nuclei, sp data only', size=20, img_key='hires', legend_fontsize=5,
              spot_size=1, frameon=False, )
sp_adata.obs['leiden_nucleus'] = sp_adata_sponly.obs['leiden']
sp_adata.obsm['X_PCA_nucleus'] = sp_adata_sponly.obsm['X_pca']
sp_adata.obsm['X_umap_nucleus'] = sp_adata_sponly.obsm['X_umap']

In [ ]:
sc.tl.dendrogram(sp_adata_sponly, groupby='leiden')
sc.tl.rank_genes_groups(sp_adata_sponly, 'leiden', method='wilcoxon', use_raw=False)

In [48]:
# Cluster-specific gene expression
pd.DataFrame(sp_adata_sponly.uns['rank_genes_groups']['names']).head(20)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
0,Myh3,Hbb-bs,Mylpf,Mylpf,Col1a2,Col4a1,Cma1,Col3a1,Hbb-bs,Sfrp2,Bgn,Col1a1,Col2a1,Col2a1,Hbb-bs,Sfrp2,Col2a1,Crabp1,Col12a1,Ibsp,Krt5,Krt1
1,Fn1,Hba-a2,Tpm2,Acta1,Col1a1,Cav1,Col3a1,Vim,Hba-a2,Col3a1,Col1a1,Col1a2,Col9a3,Col9a3,Col3a1,Crabp1,Col11a2,Col1a1,Col1a1,Col1a1,Krt1,Krt10
2,Col1a1,Mylpf,Acta1,Tpm2,Col3a1,Cald1,Cpa3,Akap12,Hbb-bt,Col1a2,Col1a2,Bgn,Col9a2,Col9a2,Col1a1,Kctd12,Hapln1,Col3a1,Col1a2,Col1a2,Cxcl14,Krtdap
3,Ttn,Acta1,Tnnc1,Ttn,Sparc,Tmsb4x,Col1a1,Kctd12,Col4a1,Col1a1,Col3a1,Tnmd,Col11a2,Col11a2,Col1a2,Ptn,Col9a3,Col1a2,Col3a1,Alpl,Krt14,Dsp
4,Col1a2,Tpm2,Tnnt1,Myh3,Postn,Acta2,Col1a2,Postn,Cav1,Ptn,Sparc,Sparc,Col9a1,Matn1,Hba-a2,Igfbp4,Col10a1,Kctd12,Sparc,Sgms2,Fabp5,Dmkn
5,Tpm2,Myh3,Ttn,Tnnc1,Dcn,Tagln,Tpsb2,Tmsb4x,Tmsb4x,Igfbp4,Postn,Col12a1,Matn1,Col9a1,Sparc,Col3a1,Sgms2,Dcn,Kctd12,Sparc,Krt10,Lor
6,Mef2c,Ttn,Actc1,Myh8,Vim,Rgs5,Dcn,Col1a1,Alas2,Postn,Kctd12,Fmod,Cnmd,Acan,Vim,Postn,Col9a2,Igfbp4,Igfbp4,Spp1,Dsp,Hrnr
7,Thbs4,Tnnt1,Myh3,Tnnt1,Kctd12,Vim,Hdc,Mest,Vim,Kctd12,Igfbp4,Serpinh1,Acan,Cnmd,Dcn,Gpc3,Acan,Igfbp5,Col5a1,Serpinh1,Ptprf,Fabp5
8,Mest,Tnnc1,Tnni1,Actc1,Col5a1,Pecam1,Sparc,Apoe,Egfl7,Fbn2,Mest,Postn,Col11a1,Col11a1,Akap12,Fstl1,Col11a1,Sparc,Postn,Pcolce,Gja1,Krt77
9,Mylpf,Actc1,Tnnt2,Tnni1,Igfbp4,Gucy1a1,Srgn,Sfrp2,Sparc,Sparc,Serpinh1,Thbs2,Wwp2,Col27a1,Col5a1,Marcks,Comp,Marcks,Igfbp5,Serpine2,Krtdap,Lgals7


In [50]:
huetracer.plot.name_clusters_interactively(
    sp_adata_sponly
)

In [ ]:
sc.pl.spatial(sp_adata_sponly, color='predicted_cell_type',
              title='Annotation for nuclei, sp data only', size=20, img_key='hires', legend_fontsize=5,
              spot_size=1, frameon=False)


In [ ]:
# Save spatial data for cell-cell interaction analysis 
sp_adata_sponly.write_h5ad(h5ad_predicted_full_save_path)